# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [23]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
import importlib

# Force reload of utils module to prevent caching
if 'utils' in sys.modules:
    del sys.modules['utils']

# Enable automatic reloading of modules
%load_ext autoreload
%autoreload 2

# Import utility functions (will be freshly loaded)
import utils
from utils import (
    load_and_clean_data, 
    calculate_profitable_trades, 
    analyze_entry_timing, 
    display_profitable_strategies,
    Strategy,
    create_strategy_library,
    evaluate_all_strategies,
    get_top_strategies,
    get_top_strategies_by_edge,
    analyze_pullback_profitability
)

# Force reload utils to ensure latest version
importlib.reload(utils)

# Load the data
df = load_and_clean_data()

# Create comparison table for BOS and CH trades
# Using 'BOS/CH' column which is the correct column name
bos_trades = df[df['BOS/CH'] == 'BOS'].copy()
ch_trades = df[df['BOS/CH'] == 'CH'].copy()

# Calculate profitability for regular trades (TP > SL)
bos_regular_profitable = (bos_trades['TP'] > bos_trades['SL']).sum()
bos_regular_total = len(bos_trades)
bos_regular_percentage = (bos_regular_profitable / bos_regular_total * 100) if bos_regular_total > 0 else 0

ch_regular_profitable = (ch_trades['TP'] > ch_trades['SL']).sum()
ch_regular_total = len(ch_trades)
ch_regular_percentage = (ch_regular_profitable / ch_regular_total * 100) if ch_regular_total > 0 else 0

# Calculate profitability for "With Extra" trades 
# Based on the utils.py logic, "With Extra" means either:
# - SL != Pullback (standard trade)
# - OR (SL == Pullback AND Extra < 1) (trade with less than 1 pip extra)
bos_with_extra = bos_trades[((bos_trades['SL'] != bos_trades['Pullback']) | ((bos_trades['SL'] == bos_trades['Pullback']) & (bos_trades['Extra'] < 1)))]
bos_extra_profitable = (bos_with_extra['TP'] > (bos_with_extra['SL'] + 1)).sum()
bos_extra_total = len(bos_with_extra)
bos_extra_percentage = (bos_extra_profitable / bos_extra_total * 100) if bos_extra_total > 0 else 0

ch_with_extra = ch_trades[((ch_trades['SL'] != ch_trades['Pullback']) | ((ch_trades['SL'] == ch_trades['Pullback']) & (ch_trades['Extra'] < 1)))]
ch_extra_profitable = (ch_with_extra['TP'] > (ch_with_extra['SL'] + 1)).sum()
ch_extra_total = len(ch_with_extra)
ch_extra_percentage = (ch_extra_profitable / ch_extra_total * 100) if ch_extra_total > 0 else 0

# Create comparison dataframe
comparison_data = {
    'Trade Type': ['BOS - Regular', 'BOS - With Extra (1 pip)', 'CH - Regular', 'CH - With Extra (1 pip)'],
    'Total Trades': [bos_regular_total, bos_extra_total, ch_regular_total, ch_extra_total],
    'Profitable Trades': [bos_regular_profitable, bos_extra_profitable, ch_regular_profitable, ch_extra_profitable],
    'Win Rate': [f'{bos_regular_percentage:.1f}%', f'{bos_extra_percentage:.1f}%', 
                 f'{ch_regular_percentage:.1f}%', f'{ch_extra_percentage:.1f}%']
}

# Additional breakdown showing BOS vs CH distribution
display(HTML("""
    <h2>Trade Type Distribution</h2>
    <p>Number of BOS and CH trades and their profitability breakdown:</p>
"""))

distribution_data = {
    'Structure Type': ['BOS', 'CH'],
    'Total Trades': [bos_regular_total, ch_regular_total],
    'Profitable': [bos_regular_profitable, ch_regular_profitable],
    'Losing': [bos_regular_total - bos_regular_profitable, ch_regular_total - ch_regular_profitable],
    'Win Rate %': [f'{bos_regular_percentage:.1f}%', f'{ch_regular_percentage:.1f}%']
}

distribution_df = pd.DataFrame(distribution_data)
display(distribution_df)

display(HTML("""
    <p><b>Key Insights:</b></p>
    <ul>
        <li>BOS trades show significantly better performance than CH trades</li>
        <li>CH trades have a notably lower win rate, confirming they should be avoided</li>
    </ul>
"""))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Structure Type,Total Trades,Profitable,Losing,Win Rate %
0,BOS,88,44,44,50.0%
1,CH,52,21,31,40.4%


In [24]:
# Display entry timing analysis using the imported function
display(HTML("""
    <h2>When To Enter</h2>
    <p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>
"""))
display(analyze_entry_timing(df))
display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>Very interesting that here, my strategy with 1M confirmation candle is the worse.</li>
"""))
display(HTML("""
    <p><b>Open Questions</b></p>
    <ul>
        <li>That 5M Stop entry makes a lot of sense as all profitable strategies would have it, but in reality somehow it performs the worse. Why?</li>
        <li>Why does 1M confirmation candle entry outperforms all other entries in the next setups analysis? Maybe because it gives worse RRR but still profitable?</li>
    </ul>
"""))

,Idea,Notation,Win Rate,With Extra
0,5M Stop,42W - 47L,47.2%,52.8%
1,5M Breakout,40W - 52L,43.5%,48.9%
2,5M Confirmation Candle,41W - 56L,42.3%,47.4%
3,1M Confirmation Candle,50W - 90L,35.7%,39.3%


In [25]:
# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

# Add all strategies from the library
strategies.extend(create_strategy_library())

# Evaluate all strategies
strategy_results = evaluate_all_strategies(df, strategies)

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

# Display top performing strategies for each RRR - sorted by Edge
display(HTML("<h2>🎯 TOP Performing Strategies</h2>"))

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies by Edge</h3>"))
    
    # Get and display top strategies sorted by edge
    top_df = get_top_strategies_by_edge(strategy_results, rrr_column)
    top_df = top_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px'}
    ).set_properties(
        subset=['Edge'],
        **{'color': 'green'}
    )
    
    display(styled_df)
    print()  # Add spacing

display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>EMA + BOS is on 5th place. So anything above it should be tradeable</li>
        <li>According this list, trading all BOS trades would simplify things a lot. Lets try that out!</li>
    </ul>
"""))

,Top 1:1 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,30M Trend + 5 < SL < 15,1M CC,28,16,12,57.1%,7.1%,4R
1,30M Trend + SL > 5 pips,1M CC,29,16,13,55.2%,5.2%,3R
2,Aggressive: SL >= 7 pips,1M CC,24,13,11,54.2%,4.2%,2R
3,30M Trend + BOS + SL < 10,1M CC,58,31,27,53.4%,3.4%,4R
4,30M Trend + News > 2hrs,1M CC,19,10,9,52.6%,2.6%,1R
5,30M Trend + BOS,1M CC,60,31,29,51.7%,1.7%,2R
6,30M Trend + 3 < SL < 10,1M CC,59,30,29,50.8%,0.8%,1R


,Top 1:2 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,10,5,5,50.0%,16.7%,5R
1,30M Trend + BOS + SL < 10,1M CC,58,24,34,41.4%,8.1%,14R
2,30M Trend + BOS,1M CC,60,24,36,40.0%,6.7%,12R
3,30M Trend + EMA + BOS + SL < 10,1M CC,50,19,31,38.0%,4.7%,7R
4,BOS,1M CC,88,33,55,37.5%,4.2%,11R
5,30M Trend + News > 2hrs,1M CC,19,7,12,36.8%,3.5%,2R
6,30M Trend + EMA + BOS,1M CC,52,19,33,36.5%,3.2%,5R
7,30M Trend + No News,1M CC,55,20,35,36.4%,3.1%,5R
8,30M Trend + SL < 10 pips,1M CC,83,30,53,36.1%,2.8%,7R
9,News in 2+ Hours,1M CC,36,13,23,36.1%,2.8%,3R


,Top 1:3 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,10,5,5,50.0%,25.0%,10R
1,Conservative: SL <= 2 pips,1M CC,14,5,9,35.7%,10.7%,6R
2,30M Trend + BOS + SL < 10,1M CC,58,19,39,32.8%,7.8%,18R
3,30M Trend + EMA + BOS + SL < 10,1M CC,50,16,34,32.0%,7.0%,14R
4,30M Trend + BOS,1M CC,60,19,41,31.7%,6.7%,16R
5,30M Trend + News > 2hrs,1M CC,19,6,13,31.6%,6.6%,5R
6,30M Trend + EMA + BOS,1M CC,52,16,36,30.8%,5.8%,12R
7,BOS,1M CC,88,25,63,28.4%,3.4%,12R
8,30M Trend + EMA + SL < 10,1M CC,64,18,46,28.1%,3.1%,8R
9,News in 2+ Hours,1M CC,36,10,26,27.8%,2.8%,4R


In [26]:
# Pullback Analysis
display(HTML("<h2>Pullback Impact on Profitability</h2>"))
display(HTML("<p>How does pullback size affect trade profitability? Analyzing trades where TP > SL.</p>"))

# Display pullback profitability analysis
pullback_analysis = analyze_pullback_profitability(df)
display(pullback_analysis)

# Add summary
display(HTML("""
<p><b>Summary</b></p>
<ul>
    <li>According to this analysis, pullback of 0.5 pip should not harm the strategy but could dramatically reduce the the path to the TP.</li>
</ul>
"""))

,Pullback,All Trades,Profitable Trades,Percentage
0,All (No Filter),140,65,46.4%
1,0.5 pips,133,60,45.1%
2,1.0 pip,124,51,41.1%
3,1.5 pips,115,44,38.3%
4,2.0 pips,97,35,36.1%
5,2.5 pips,87,28,32.2%
6,3.0 pips,78,23,29.5%


In [27]:
# Display only profitable strategies
display_profitable_strategies(strategy_results)

,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,24,24,24
1,Wins,13,6,4
2,Losses,11,18,20
3,Win Rate,54.2%,25.0%,16.7%
4,Edge,4.2%,-8.3%,-8.3%
5,Outcome,2R,-6R,-8R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + BOS,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,60,60,60
1,Wins,31,24,19
2,Losses,29,36,41
3,Win Rate,51.7%,40.0%,31.7%
4,Edge,1.7%,6.7%,6.7%
5,Outcome,2R,12R,16R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + SL > 5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,29,29,29
1,Wins,16,10,6
2,Losses,13,19,23
3,Win Rate,55.2%,34.5%,20.7%
4,Edge,5.2%,1.2%,-4.3%
5,Outcome,3R,1R,-5R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + 3 < SL < 10,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,59,59,59
1,Wins,30,19,12
2,Losses,29,40,47
3,Win Rate,50.8%,32.2%,20.3%
4,Edge,0.8%,-1.1%,-4.7%
5,Outcome,1R,-2R,-11R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,28,28,28
1,Wins,16,10,6
2,Losses,12,18,22
3,Win Rate,57.1%,35.7%,21.4%
4,Edge,7.1%,2.4%,-3.6%
5,Outcome,4R,2R,-4R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + BOS + SL < 10,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,58,58,58
1,Wins,31,24,19
2,Losses,27,34,39
3,Win Rate,53.4%,41.4%,32.8%
4,Edge,3.4%,8.1%,7.8%
5,Outcome,4R,14R,18R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + News > 2hrs,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,19,19,19
1,Wins,10,7,6
2,Losses,9,12,13
3,Win Rate,52.6%,36.8%,31.6%
4,Edge,2.6%,3.5%,6.6%
5,Outcome,1R,2R,5R
6,Entry,1M CC,1M CC,1M CC
